# Abschnitt 3: k-NN-Klassifikation von Muenzen

Dieses Notebook ist fuer **Google Colab** gedacht. Es laedt CSV-Dateien aus dem Dashboard, extrahiert Peak-Merkmale von Muenzen oder muenzaehnlichen Objekten und trainiert ein k-Nearest-Neighbors-Modell.

Am Ende wird eine `model.json` erzeugt, die im Dashboard auf den ESP32 hochgeladen werden kann.

## 1. Bibliotheken importieren

Wir verwenden `pandas` fuer Tabellen, `numpy` fuer Rechnungen und `scikit-learn` fuer k-NN.

In [ ]:
from __future__ import annotations

import io
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

FEATURES = ["rp_min", "l_min", "delta_rp", "delta_l", "rp_avg", "l_avg"]

# Servo-Zielwinkel fuer den Live-Test am Sortierer.
# Diese Werte muessen an den mechanischen Aufbau angepasst werden.
CLASS_ANGLES = {
    "objekt_a": 25,
    "objekt_b": 45,
    "objekt_c": 65,
    "objekt_d": 85,
    "objekt_e": 105,
    "objekt_f": 125,
}

print("Setup fertig")

from sklearn.neighbors import KNeighborsClassifier


## 2. CSV-Dateien hochladen

Lade hier die CSV-Dateien aus dem Dashboard hoch. Sinnvolle Dateinamen sind zum Beispiel:

- `objekt_a.csv`
- `messing_chip.csv`
- `stahl_scheibe.csv`

Das Label wird aus dem Dateinamen abgeleitet.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    CSV_FILES = {name: data for name, data in uploaded.items() if name.lower().endswith(".csv")}
except Exception:
    # Fallback für lokale Jupyter-Umgebungen:
    # CSV-Dateien in den Ordner messdaten/ legen.
    CSV_FILES = {p.name: p.read_bytes() for p in Path("messdaten").glob("*.csv")}

print(f"{len(CSV_FILES)} CSV-Dateien geladen")
list(CSV_FILES.keys())[:10]

## 3. Hilfsfunktionen

Die nächsten Funktionen übernehmen drei Aufgaben:

- Label aus Dateinamen bestimmen
- CSV-Dateien einlesen
- Peaks finden und daraus Merkmale berechnen

Die Peak-Erkennung ist bewusst einfach gehalten: Ein Peak beginnt, wenn `RP` deutlich unter die Luftreferenz fällt, und endet, wenn das Signal zurückkehrt.

In [ ]:
def label_from_filename(filename: str) -> str:
    """Leitet das Klassenlabel aus dem Dateinamen ab."""
    stem = Path(filename).stem.lower()
    for token in ("muenze_", "munze_", "coin_"):
        stem = stem.replace(token, "")
    return stem


def read_dashboard_csv(raw: bytes) -> pd.DataFrame:
    """Liest Dashboard-CSV ein. Neuere Exporte nutzen Semikolon, alte ggf. Komma."""
    text = raw.decode("utf-8-sig")
    df = pd.read_csv(io.StringIO(text), sep=";")
    if len(df.columns) == 1:
        df = pd.read_csv(io.StringIO(text))
    df = df.rename(columns=str.strip)
    required_raw = {"Zeit_ms", "RP", "L"}
    required_peak = {"rp_min", "l_min", "delta_rp", "delta_l"}
    if required_raw <= set(df.columns): return df  # Rohdaten-CSV
    if required_peak <= set(df.columns): return df  # Peak-CSV
    raise ValueError(f"CSV-Spalten fehlen: {required_raw} oder {required_peak}")


def is_peak_csv(df: pd.DataFrame) -> bool:
    return {"rp_min", "l_min", "delta_rp", "delta_l"} <= set(df.columns)


def extract_peaks(df: pd.DataFrame, label: str, min_gap: int = 3) -> list[dict]:
    """Extrahiert Peak-Features aus einer Messreihe."""
    rp = df["RP"].to_numpy(dtype=float)
    l_val = df["L"].to_numpy(dtype=float)

    # Robuste Luftreferenz: hoher RP-Wert und niedriger L-Wert entsprechen typischerweise Luft.
    rp_air = float(np.percentile(rp, 90))
    l_air = float(np.percentile(l_val, 90))

    # Ein Objekt wird angenommen, wenn RP deutlich unter die Luftreferenz fällt.
    baseline_window = rp[: min(len(rp), 50)]
    noise = max(float(np.std(baseline_window)), 20.0)
    threshold = rp_air - 4.0 * noise

    peaks = []
    active = False
    start = 0
    gap = 0

    for i, value in enumerate(rp):
        present = value < threshold

        if present and not active:
            active = True
            start = i
            gap = 0
        elif active and not present:
            gap += 1
            if gap >= min_gap:
                end = max(start + 1, i - gap + 1)
                seg_rp = rp[start:end]
                seg_l = l_val[start:end]

                # Sehr kurze Segmente werden ignoriert, weil sie meist Rauschen sind.
                if len(seg_rp) >= 3:
                    rp_min = float(np.min(seg_rp))
                    l_min = float(np.min(seg_l))
                    peaks.append({
                        "label": label,
                        "rp_min": rp_min,
                        "l_min": l_min,
                        "delta_rp": rp_air - rp_min,
                        "delta_l": l_air - l_min,
                        "rp_avg": float(np.mean(seg_rp)),
                        "l_avg": float(np.mean(seg_l)),
                    })
                active = False
        elif active:
            gap = 0

    return peaks

## 4. Peaks extrahieren

Aus jeder CSV-Datei werden jetzt Peaks extrahiert. Die resultierende Tabelle enthält eine Zeile pro Münzdurchlauf.

In [ ]:
rows = []
raw_tables = {}

for filename, raw in CSV_FILES.items():
    label = label_from_filename(filename)
    df = read_dashboard_csv(raw)
    raw_tables[filename] = df
    if is_peak_csv(df):
        for _, row in df.iterrows():
            rows.append({"label": label, "rp_min": row["rp_min"], "l_min": row["l_min"], "delta_rp": row["delta_rp"], "delta_l": row["delta_l"], "rp_avg": row.get("rp_avg", row["rp_min"]), "l_avg": row.get("l_avg", row["l_min"])})
    else:
        rows.extend(extract_peaks(df, label))

data = pd.DataFrame(rows)

print(f"{len(data)} Peaks extrahiert")
display(data.head())
display(data.groupby("label").size().rename("Peaks pro Klasse"))

## 5. Streudiagramm der Merkmale

Das Streudiagramm hilft, manuelle Grenzen abzuleiten und zu erkennen, welche Klassen gut oder schlecht trennbar sind.

In [ ]:
plt.figure(figsize=(7, 5))
for label, group in data.groupby("label"):
    plt.scatter(group["delta_rp"], group["delta_l"], label=label, alpha=0.8)

plt.xlabel("delta_rp")
plt.ylabel("delta_l")
plt.title("Peak-Features der Münzen")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 6. Trainings- und Testdaten bilden

Wir teilen die Daten in Trainingsdaten und Testdaten. Das Modell wird nur mit den Trainingsdaten gebaut. Die Testdaten bleiben zurück, um zu prüfen, wie gut das Modell auf neuen Messungen funktioniert.

In [ ]:
X = data[FEATURES].to_numpy()
y = data["label"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# k-NN und Entscheidungsbaum arbeiten besser vergleichbar, wenn alle Features skaliert sind.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Training: {len(X_train)} Peaks")
print(f"Test:     {len(X_test)} Peaks")

## 7. k-NN trainieren und vergleichen

k-NN klassifiziert eine neue Messung anhand der naechsten Trainingspunkte im Merkmalsraum. Der Parameter `k` bestimmt, wie viele Nachbarn abstimmen. Kleine `k` reagieren empfindlicher auf Ausreisser, groessere `k` glaetten staerker.

In [ ]:
for k in (1, 3, 5):
    knn_tmp = KNeighborsClassifier(n_neighbors=k)
    knn_tmp.fit(X_train_s, y_train)
    print(f"k={k}: Genauigkeit = {knn_tmp.score(X_test_s, y_test):.3f}")

# Fuer den Export verwenden wir k=3 als gut erklaerbaren Standard.
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_s, y_train)
knn_pred = knn.predict(X_test_s)

print(classification_report(y_test, knn_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, knn_pred)
plt.title("Konfusionsmatrix: k-NN")
plt.xticks(rotation=45)
plt.show()


Vergleich verschiedener Feature-Subsets — welche Features reichen aus?

In [ ]:
# ── Feature-Subset-Vergleich ──
# Welche 2-4 Features reichen für gute Trennung?
subsets = {
    "rp_min + l_min":           ["rp_min", "l_min"],
    "delta_rp + delta_l":       ["delta_rp", "delta_l"],
    "rp_avg + l_avg":           ["rp_avg", "l_avg"],
    "alle 4 (ohne avg)":        ["rp_min", "l_min", "delta_rp", "delta_l"],
    "alle 6":                   ["rp_min", "l_min", "delta_rp", "delta_l", "rp_avg", "l_avg"],
}
print(f"{'Subset':<30} {'Train-Acc':>10} {'Test-Acc':>10}")
print("-" * 52)
for name, cols in subsets.items():
    Xs_tr = scaler.transform(X_train)[:, [FEATURES.index(c) for c in cols]]
    Xs_te = scaler.transform(X_test)[:, [FEATURES.index(c) for c in cols]]
    from sklearn.neighbors import KNeighborsClassifier as _KNN
    m = _KNN(n_neighbors=3).fit(Xs_tr, y_train)
    print(f"{name:<30} {m.score(Xs_tr, y_train):>10.3f} {m.score(Xs_te, y_test):>10.3f}")


## 8. k-NN-Modell fuer ESP32 exportieren

Die Firmware erwartet normalisierte Trainingssamples. Darum speichern wir Mittelwert, Standardabweichung und alle Trainingspunkte in `model.json`. Der Baum-Teil bleibt leer.

In [ ]:
model = {
    "version": 2,
    "features": FEATURES,
    "classes": [
        {"label": str(label), "angle": CLASS_ANGLES.get(str(label), 90)}
        for label in sorted(set(y))
    ],
    "manual": {"rules": []},
    "knn": {
        "k": 3,
        "mean": scaler.mean_.tolist(),
        "std": scaler.scale_.tolist(),
        "samples": [
            {
                "label": str(label),
                "angle": CLASS_ANGLES.get(str(label), 90),
                "x": x.tolist(),
            }
            for x, label in zip(X_train_s, y_train)
        ],
    },
    "tree": {"nodes": []},
}

with open("model.json", "w", encoding="utf-8") as f:
    json.dump(model, f, indent=2)

print("model.json fuer k-NN geschrieben")


## 11. `model.json` herunterladen

In Colab lädt die nächste Zelle die Datei direkt herunter. Anschließend kann sie im Dashboard unter `model.json hochladen` an den ESP32 übertragen werden.

In [ ]:
try:
    from google.colab import files
    files.download("model.json")
except Exception:
    print("Lokale Umgebung: model.json liegt im aktuellen Arbeitsordner.")